## Reading artifacts & features

In [1]:
import pandas as pd
import numpy as np

# Load artifacts
train_df = pd.read_csv('/content/train.csv')
test_df  = pd.read_csv('/content/test.csv')

# Separate features and target
X_train = train_df.drop(columns=['price'])
y_train = train_df['price']

X_test  = test_df.drop(columns=['price'])
y_test  = test_df['price']

print("Train:", X_train.shape)
print("Test: ", X_test.shape)
print("\nFeatures:", X_train.columns.tolist())

Train: (43035, 10)
Test:  (10759, 10)

Features: ['carat', 'cut', 'color', 'clarity', 'depth', 'table', 'zirconia_length', 'zirconia_width', 'zirconia_height', 'volume']


In [2]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor
from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

# Evaluation function — reusable for all models
def evaluate_model(name, model, X_train, y_train, X_test, y_test):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    r2   = r2_score(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae  = mean_absolute_error(y_test, y_pred)

    return {
        'Model': name,
        'R2 Score': round(r2, 4),
        'RMSE':     round(rmse, 4),
        'MAE':      round(mae, 4),
        'Fitted':   model        # keep model object for later
    }

### Define models to try

In [3]:
# Define all models to try
models = {
    'Linear Regression':    LinearRegression(),
    'Ridge':                Ridge(),
    'Lasso':                Lasso(),
    'Decision Tree':        DecisionTreeRegressor(random_state=42),
    'Random Forest':        RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting':    GradientBoostingRegressor(random_state=42),
    'XGBoost':              XGBRegressor(random_state=42, verbosity=0)
}

# Train and evaluate all models
results = []
fitted_models = {}

for name, model in models.items():
    print(f"Training {name}...")
    result = evaluate_model(name, model, X_train, y_train, X_test, y_test)
    fitted_models[name] = result.pop('Fitted')  # store separately
    results.append(result)

print("\nDone!")

Training Linear Regression...
Training Ridge...
Training Lasso...
Training Decision Tree...
Training Random Forest...
Training Gradient Boosting...
Training XGBoost...

Done!


In [4]:
# Display results — sorted by R2 Score
results_df = pd.DataFrame(results).sort_values('R2 Score', ascending=False)
results_df = results_df.reset_index(drop=True)
results_df.index += 1  # start ranking from 1
print(results_df.to_string())

               Model  R2 Score    RMSE     MAE
1            XGBoost    0.9921  0.0891  0.0638
2      Random Forest    0.9916  0.0918  0.0634
3  Gradient Boosting    0.9884  0.1078  0.0819
4      Decision Tree    0.9848  0.1236  0.0840
5              Ridge    0.9570  0.2074  0.1137
6  Linear Regression    0.9567  0.2082  0.1137
7              Lasso    0.1564  0.9193  0.7909


## XGBOOST AND RANDOM FOREST PERFORMANCE ARE CLOSE

## Fine tune XGBOOST

In [5]:
from sklearn.model_selection import RandomizedSearchCV

# Parameter grid to search
param_grid = {
    'n_estimators':      [100, 200, 300, 500],
    'max_depth':         [3, 4, 5, 6, 7, 8],
    'learning_rate':     [0.01, 0.05, 0.1, 0.2, 0.3],
    'subsample':         [0.6, 0.7, 0.8, 0.9, 1.0],
    'colsample_bytree':  [0.6, 0.7, 0.8, 0.9, 1.0],
    'min_child_weight':  [1, 3, 5, 7],
    'gamma':             [0, 0.1, 0.2, 0.3]
}

## Fine tune RandomForest

In [6]:
# RandomizedSearchCV — faster than GridSearch, still thorough
xgb_base = XGBRegressor(random_state=42, verbosity=0)

random_search = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=param_grid,
    n_iter=50,               # tries 50 random combinations
    scoring='r2',
    cv=5,                    # 5-fold cross validation
    random_state=42,
    n_jobs=-1,               # use all CPU cores
    verbose=1
)

print("Starting RandomizedSearchCV — this may take 3-5 minutes...")
random_search.fit(X_train, y_train)
print("\nDone!")

Starting RandomizedSearchCV — this may take 3-5 minutes...
Fitting 5 folds for each of 50 candidates, totalling 250 fits

Done!


## Examine Best parameter for random forest

In [7]:
# Best parameters found
print("Best Parameters:")
for param, value in random_search.best_params_.items():
    print(f"  {param}: {value}")

print(f"\nBest CV R2 Score: {random_search.best_score_:.4f}")

Best Parameters:
  subsample: 0.7
  n_estimators: 500
  min_child_weight: 5
  max_depth: 8
  learning_rate: 0.1
  gamma: 0
  colsample_bytree: 0.7

Best CV R2 Score: 0.9927


## Compare Baseline XGboost to Fine tuned XGboost

In [8]:
# Evaluate tuned model on test set
best_xgb = random_search.best_estimator_

y_pred_tuned = best_xgb.predict(X_test)

r2_tuned   = r2_score(y_test, y_pred_tuned)
rmse_tuned = np.sqrt(mean_squared_error(y_test, y_pred_tuned))
mae_tuned  = mean_absolute_error(y_test, y_pred_tuned)

print("="*45)
print("TUNED XGBoost vs BASELINE XGBoost")
print("="*45)
print(f"{'Metric':<10} {'Baseline':>12} {'Tuned':>12} {'Improved':>10}")
print("-"*45)
print(f"{'R2':<10} {0.9921:>12} {r2_tuned:>12.4f} {'+' if r2_tuned > 0.9921 else '-'}")
print(f"{'RMSE':<10} {0.0891:>12} {rmse_tuned:>12.4f} {'+' if rmse_tuned < 0.0891 else '-'}")
print(f"{'MAE':<10} {0.0638:>12} {mae_tuned:>12.4f} {'+' if mae_tuned < 0.0638 else '-'}")

TUNED XGBoost vs BASELINE XGBoost
Metric         Baseline        Tuned   Improved
---------------------------------------------
R2               0.9921       0.9926 +
RMSE             0.0891       0.0861 +
MAE              0.0638       0.0601 +


## Save best model

In [9]:
import pickle
import os

# Save tuned XGBoost model
os.makedirs('artifacts', exist_ok=True)

with open('artifacts/model.pkl', 'wb') as f:
    pickle.dump(best_xgb, f)

print("✅ Model saved to artifacts/model.pkl")

✅ Model saved to artifacts/model.pkl


Verify it saves and loads correctly

In [10]:
# Verify it saves and loads correctly
with open('artifacts/model.pkl', 'rb') as f:
    loaded_model = pickle.load(f)

# Quick sanity check — predictions should match
y_pred_loaded = loaded_model.predict(X_test)
print(f"Load verification R2: {r2_score(y_test, y_pred_loaded):.4f}")
print("✅ Model loads and predicts correctly!" if r2_score(y_test, y_pred_loaded) == r2_tuned else "❌ Mismatch!")

Load verification R2: 0.9926
✅ Model loads and predicts correctly!


## modeling summary

In [11]:
# Final modeling notebook summary
print("="*50)
print("MODELING NOTEBOOK — COMPLETE")
print("="*50)
print(f"\nBest Model:    Tuned XGBoost")
print(f"R2 Score:      {r2_tuned:.4f}")
print(f"RMSE:          {rmse_tuned:.4f}")
print(f"MAE:           {mae_tuned:.4f}")
print(f"\nBest Params:")
for param, value in random_search.best_params_.items():
    print(f"  {param}: {value}")
print(f"\nSaved to:  artifacts/model.pkl")

MODELING NOTEBOOK — COMPLETE

Best Model:    Tuned XGBoost
R2 Score:      0.9926
RMSE:          0.0861
MAE:           0.0601

Best Params:
  subsample: 0.7
  n_estimators: 500
  min_child_weight: 5
  max_depth: 8
  learning_rate: 0.1
  gamma: 0
  colsample_bytree: 0.7

Saved to:  artifacts/model.pkl
